# TriForge on Colab: one session of a 2B ternary fine-tune (free T4)

1. **Runtime → Change runtime type → T4 GPU.**
2. **Secrets** (key icon in the left sidebar): add `GITHUB_TOKEN`, a GitHub token that can read
   Vedant-Sood/TernaryAI, and switch on *Notebook access*. Optionally `HF_TOKEN`.
3. **Runtime → Run all**, and allow Google Drive access when asked.

Keep this tab open: the free tier stops a session when the browser is closed or idle for long.
Checkpoints are copied to `MyDrive/TriForge` as soon as they are written, so a disconnect loses at
most the steps since the last checkpoint. To continue, open the notebook again and **Run all**:
it resumes from the copy in Drive. See `docs/CLOUD_GPU.md` in the repo.

In [ ]:
import os, time
os.environ["TRIFORGE_SESSION_START"] = str(time.time())   # the training cap is computed from this

SPECIALIST = "trimath"   # tricode | trimath | trireason
PRESET = "quick"         # quick (~30 min, measures speed) | partial (full run, several sessions)
BRANCH = "claude/project-thread-psmmpb"   # switch to "main" once the TriForge PR is merged
REPO = "github.com/Vedant-Sood/TernaryAI.git"
WORK = "/content/TernaryAI"
DRIVE = "/content/drive/MyDrive/TriForge"   # checkpoints, data and exports live here

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE, exist_ok=True)
!df -h /content/drive | tail -1

In [ ]:
from google.colab import userdata
token = ""
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception as e:
    print(f"no GITHUB_TOKEN secret ({type(e).__name__}): cloning without a token (works only if the repo is public)")
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

import subprocess
url = f"https://x-access-token:{token}@{REPO}" if token else f"https://{REPO}"
if os.path.isdir(WORK):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only", url, BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, WORK], check=True)
# never leave the token in .git/config
subprocess.run(["git", "-C", WORK, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
del token, url
%cd {WORK}
!git log --oneline -1

In [ ]:
# The preinstalled CUDA torch is kept; only the other dependencies are installed.
!pip install -q -r requirements/base.txt -r requirements/hf.txt "bitsandbytes>=0.43"
!pip install -q --no-deps -e .
!python scripts/check_env.py

In [ ]:
!python scripts/cloud_session.py --specialist {SPECIALIST} --preset {PRESET} --input-root {DRIVE} --sync-dir {DRIVE} --export-when-done

In [ ]:
!du -sh {DRIVE}/checkpoints/* {DRIVE}/data/* 2>/dev/null
!tail -n 3 checkpoints/{SPECIALIST}/bitnet2b_{PRESET}_t4/metrics.jsonl